In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE


url = "/Users/pruthvi/Desktop/ML_practical/winequality.csv"
df = pd.read_csv(url)

print(f"Dataset Shape: {df.shape}")
print(f"Class Distribution:\n{df['quality'].value_counts().sort_index()}")


df = pd.get_dummies(df, columns=['type'], drop_first=True, dtype=int)


for col in df.columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mean())

X = df.drop(columns=['quality'])
y = df['quality']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

sch = StandardScaler()
X_train_scaled = sch.fit_transform(X_train)
X_test_scaled = sch.transform(X_test)


smote = SMOTE(random_state=42, k_neighbors=3)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)


clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    criterion='entropy',
    random_state=42,
    n_jobs=-1
)
clf.fit(X_train_res, y_train_res)


y_pred = clf.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
print("\n" + "="*40)
print(f"Overall Accuracy: {acc * 100:.2f}%")
print("="*40)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=[f"Actual {c}" for c in sorted(y.unique())],
    columns=[f"Pred {c}" for c in sorted(y.unique())]
))

importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': clf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\nFeature Importances:")
print(importances.to_string(index=False))

Dataset Shape: (6497, 13)
Class Distribution:
quality
3      30
4     216
5    2138
6    2836
7    1079
8     193
9       5
Name: count, dtype: int64

Overall Accuracy: 66.62%

Classification Report:
              precision    recall  f1-score   support

           3       0.00      0.00      0.00         6
           4       0.45      0.35      0.39        43
           5       0.69      0.78      0.73       428
           6       0.70      0.65      0.67       567
           7       0.58      0.62      0.60       216
           8       0.53      0.44      0.48        39
           9       0.00      0.00      0.00         1

    accuracy                           0.67      1300
   macro avg       0.42      0.40      0.41      1300
weighted avg       0.66      0.67      0.66      1300


Confusion Matrix:
          Pred 3  Pred 4  Pred 5  Pred 6  Pred 7  Pred 8  Pred 9
Actual 3       0       2       2       1       0       1       0
Actual 4       0      15      18       9       1      

In [2]:
df.shape

(6497, 13)